# Семинар 11. Сверточные сети на CIFAR-10

Для самостоятельного разбора PyTorch: [Dive into Deep Learning](https://en.d2l.ai/chapter_preface/index.html).

PyTorch хранит тензоры и параметры слоев, автоматически вычисляет производные и выполняет операции на CPU или ускорителе. Устройство слоев и порядок обучения задаем сами.

Сначала собираем небольшой пример: Dataset → DataLoader → Sequential → обучение. Затем используем готовую сеть из блоков и сравниваем архитектуры, регуляризацию, аугментации и оптимизаторы.

Четыре ячейки «Пишем вместе» заполняем на занятии. Остальной код готов. Все модели обучаем на одном разбиении, восемь эпох с батчем 128. Для работы нужны подпапка `support/` и пакеты из `requirements_week11.txt`.

In [ ]:
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from IPython.display import display

from support.week11_utils import (
    CLASSES, load_data, channel_statistics, choose_device, seed_everything,
    synchronize, show_images, evaluate_model, classification_metrics,
    predict_probabilities, corrupt_images, show_class_metrics,
)
from support.seminar11_report import (
    show_comparison, plot_comparison, plot_training_groups, plot_robustness,
)

SEED = 42
EPOCHS = 8
BATCH_SIZE = 128
DEVICE = choose_device()
seed_everything(SEED)
torch.set_num_threads(4)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("PyTorch", torch.__version__, " | ", DEVICE)

from support.lesson11_visuals import setup_style
setup_style()

## 1. Берем несколько картинок

CIFAR-10 содержит цветные картинки 32×32 десяти классов. Одна картинка — массив HWC: высота, ширина, каналы. Значения uint8 лежат от 0 до 255. Сверточный слой ожидает CHW; перед ним переставим оси и переведем числа в float32.

Используем 12 000 изображений train, 2 000 validation и 2 000 из официального test. Средние и стандартные отклонения каналов считаем только по train. Validation выбирает лучшую эпоху; test оставляем до общего отчета.

In [ ]:
data = load_data(train_size=12000, val_size=2000, test_size=2000, seed=SEED)
train_images = data["train_images"]
train_labels = data["train_labels"]
val_images = data["val_images"]
val_labels = data["val_labels"]
test_images = data["test_images"]
test_labels = data["test_labels"]

show_images(train_images[:6], [CLASSES[label] for label in train_labels[:6]])
image = train_images[0]
print("Одна картинка:", image.shape, image.dtype, "диапазон:", image.min(), image.max())
print("Метка:", train_labels[0], CLASSES[train_labels[0]])

In [ ]:
mean, std = channel_statistics(train_images)
MEAN = torch.tensor(mean).reshape(3, 1, 1)
STD = torch.tensor(std).reshape(3, 1, 1)
print("Средние RGB:", mean.round(3), " |  стандартные отклонения:", std.round(3))

## 2. Описываем Dataset — пишем вместе

Dataset хранит картинки и метки; `__getitem__` возвращает один объект. Картинку переводим из HWC в CHW, делим на 255, затем нормируем каждый канал: `(image − MEAN) / STD`. Форма MEAN и STD равна 3×1×1: значение канала повторяется по всем его пикселям.

Контракт: `image` — float32-тензор 3×32×32, `label` — скаляр long от 0 до 9.

In [ ]:
# Пишем вместе. Верните image: float32 (3,32,32) и label: long.
class CIFARDataset(Dataset):
    def __init__(self, images, labels):
        ...

    def __len__(self):
        ...

    def __getitem__(self, index):
        ...

In [ ]:
train_dataset = CIFARDataset(train_images, train_labels)
val_dataset = CIFARDataset(val_images, val_labels)
test_dataset = CIFARDataset(test_images, test_labels)

image, label = train_dataset[0]
print("Объект:", image.shape, image.dtype, " |  метка:", label, label.dtype)
print("Размер Dataset:", len(train_dataset))

## 3. Собираем DataLoader — пишем вместе

DataLoader собирает объекты в батч B×3×32×32 и метки B. Для train включаем перемешивание; validation и test проходят в исходном порядке. Отдельный генератор фиксирует порядок train-объектов независимо от случайности внутри модели.

In [ ]:
# Пишем вместе: перемешиваем только train.
generator = torch.Generator().manual_seed(SEED)
train_loader = ...
val_loader = ...
test_loader = ...

In [ ]:
images, labels = next(iter(train_loader))
print("Батч:", images.shape, images.dtype)
print("Метки:", labels.shape, labels.dtype)
visible_images = images[:8] * STD + MEAN
visible_images = visible_images.permute(0, 2, 3, 1).numpy().clip(0, 1)
show_images(visible_images, [CLASSES[label] for label in labels[:8]], columns=4)

### Проверяем pooling на изображении CIFAR-10

Берём реальную RGB-картинку из train: 32×32 пикселя, три канала. MaxPool2d(2) оставляет максимум каждого окна 2×2, AvgPool2d(2) — его среднее. Оба слоя работают с каждым каналом отдельно; выход имеет форму 1×3×16×16.

Сначала сравниваем изображения целиком. Затем увеличиваем отмеченное окно 2×2 **канала R** и проверяем два результата на его четырёх настоящих значениях. Числа — интенсивности от 0 до 1 до стандартизации, а не новые обучаемые признаки.

У MaxPool и AvgPool нет обучаемых параметров и различия между train и eval. AdaptiveAvgPool задаёт конечную форму карты; выход (1,1) соответствует глобальному среднему.

In [ ]:
from support.lesson11_visuals import plot_cifar_pooling
from support.week11_utils import draw_matrices

pool_example = torch.tensor(train_images[0].copy()).permute(2, 0, 1).float() / 255
pool_example = pool_example.unsqueeze(0)
max_layer = nn.MaxPool2d(kernel_size=2, stride=2)
avg_layer = nn.AvgPool2d(kernel_size=2, stride=2)
max_values = max_layer(pool_example)
avg_values = avg_layer(pool_example)

max_layer.eval()
avg_layer.eval()
assert torch.equal(max_layer(pool_example), max_values)
assert torch.equal(avg_layer(pool_example), avg_values)
assert max_values.shape == avg_values.shape == (1, 3, 16, 16)

row, column = 18, 10
red_window = pool_example[0, 0, row:row + 2, column:column + 2]
red_max = max_values[0, 0, row // 2, column // 2]
red_avg = avg_values[0, 0, row // 2, column // 2]
assert torch.allclose(red_max, red_window.max())
assert torch.allclose(red_avg, red_window.mean())

plot_cifar_pooling(pool_example[0].numpy(), max_values[0].numpy(), avg_values[0].numpy(), (row, column));
draw_matrices([red_window.numpy(), red_max.numpy(), red_avg.numpy()],
              ["Увеличенное окно 2x2  |  канал R", "Максимум этого окна", "Среднее этого окна"],
              cmap="Reds", value_limits=(0, 1));

## 4. Маленькая CNN через Sequential — пишем вместе

Сначала создаем отдельные слои, затем перечисляем их в `nn.Sequential`. Свёрточная нейронная сеть (CNN) использует локальные ядра и общие веса. Два MaxPool уменьшают стороны 32 → 16 → 8. После второго блока получаем 32×8×8 = 2048 чисел; Linear переводит их в 64 признака, затем в 10 оценок классов.

Схема: Conv → ReLU → Pool → Conv → ReLU → Pool → Flatten → Linear → ReLU → Linear. Softmax перед CrossEntropyLoss не нужен: функция потерь сама работает с оценками классов.

In [ ]:
# Пишем вместе: 3->16->32 канала, два Pool, затем 2048->64->10.
seed_everything(SEED)
conv1 = ...
activation1 = ...
pool1 = ...
conv2 = ...
activation2 = ...
pool2 = ...
flatten = ...
linear1 = ...
activation3 = ...
linear2 = ...

small_cnn = ...  # Соберите nn.Sequential из созданных слоев.

In [ ]:
logits = small_cnn(images)
print(small_cnn)
print("Выход:", logits.shape)
print("Параметры:", sum(parameter.numel() for parameter in small_cnn.parameters()))

## 5. Пишем цикл обучения и обучаем первую сеть

На каждом батче выполняем: обнуление градиентов → forward → loss → backward → обновление весов. Считаем средний loss по объектам, учитывая размер последнего батча. В `train()` включается обучающее поведение BatchNorm и Dropout.

In [ ]:
# Пишем вместе. loader: images (B,3,32,32), labels (B,).
# Верните средний train-loss по всем объектам эпохи.
def train_epoch(model, loader, optimizer, criterion, device):
    ...

Для проверки нужны `eval()` и `inference_mode()`: первое меняет поведение слоев, второе отключает запись графа производных. Проверка не обновляет параметры. Следующие две функции готовы: первая оценивает validation, вторая повторяет эпохи и возвращает веса с наименьшим validation-loss.

На train BatchNorm использует среднее и дисперсию текущего батча и обновляет `running_mean`/`running_var`. На eval он берет эти сохраненные оценки и не изменяет их. Dropout в PyTorch на train обнуляет с вероятностью $p$, сохранившиеся значения делит на $1-p$; на eval возвращает вход без масштабирования. Поэтому `eval()` нужен даже внутри `inference_mode()`: отключение градиентов само по себе не меняет режим слоев.

In [ ]:
@torch.inference_mode()
def validate(model, loader, criterion, device):
    model.eval()
    loss_sum = 0.0
    correct = 0
    count = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        loss_sum += criterion(logits, labels).item() * len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += len(labels)

    mean_loss = loss_sum / count
    accuracy = correct / count
    return mean_loss, accuracy

In [ ]:
def fit_model(model, loader, optimizer, epochs=EPOCHS):
    criterion = nn.CrossEntropyLoss()
    history = []
    best_loss = float("inf")
    best_state = None
    synchronize(DEVICE)
    started = time.perf_counter()

    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, loader, optimizer, criterion, DEVICE)
        val_loss, val_accuracy = validate(model, val_loader, criterion, DEVICE)
        history.append({"epoch": epoch, "train_loss": train_loss,
                        "val_loss": val_loss, "val_accuracy": val_accuracy})

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
        print(f"{epoch:2}: train={train_loss:.3f}, val={val_loss:.3f}, acc={val_accuracy:.3f}")

    model.load_state_dict(best_state)
    model.eval()
    synchronize(DEVICE)
    seconds = time.perf_counter() - started
    return history, seconds

In [ ]:
# Восстанавливаем генератор после демонстрационного чтения батча.
train_loader.generator.manual_seed(SEED)
seed_everything(SEED)
small_cnn = small_cnn.to(DEVICE)
optimizer = torch.optim.Adam(small_cnn.parameters(), lr=.001, weight_decay=1e-4)
small_history, small_seconds = fit_model(small_cnn, train_loader, optimizer)

models = {"Sequential": small_cnn}
histories = {"Sequential": small_history}
training_times = {"Sequential": small_seconds}
plot_training_groups(histories, {"Первая обученная сеть": ["Sequential"]})

### Заглянем внутрь обученной сети

Не каждый канал является детектором понятного человеку объекта. Посмотрим, как **одна и та же картинка** представлена после первого и второго ReLU: пространственный размер меняется, а каждый канал даёт свою карту откликов.

На рисунке первые шесть каналов без отбора по красоте. Внутри строки шкала общая, между слоями — отдельная. Поэтому сравниваем расположение откликов, но не яркость разных строк. Нулевая карта одной картинки ещё не означает «мёртвый» фильтр: это проверяют на многих объектах.

Приём: [CS231n, Visualizing what ConvNets learn](https://cs231n.github.io/understanding-cnn/). Здесь визуализируются активации модели, только что обученной выше.

In [ ]:
from support.lesson11_visuals import plot_feature_atlas

small_cnn.eval()
probe_image, probe_label = test_dataset[0]
feature_maps = {}
with torch.inference_mode():
    features = probe_image.unsqueeze(0).to(DEVICE)
    for layer_index, layer in enumerate(small_cnn):
        features = layer(features)
        if isinstance(layer, nn.ReLU) and features.ndim == 4:
            feature_maps[f"После ReLU, слой {layer_index}"] = features[0].cpu().numpy()

assert len(feature_maps) == 2
assert all(values.shape[0] >= 6 for values in feature_maps.values())
plot_feature_atlas(test_images[0], feature_maps);

## 6. Готовый ConvBlock

В блоке две свертки. После каждой можно включить BatchNorm и выбрать функцию активации; в конце — Dropout2d, который обнуляет целые каналы. `stride` первой свертки задает уменьшение размера, у второй шаг равен 1.

Смещения сверток оставлены во всех вариантах: так общие веса при одинаковом seed совпадают и при включении BatchNorm. В обычной архитектуре перед BatchNorm смещения можно отключать, поскольку нормализация устраняет общий сдвиг канала.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3,
                 stride=1, padding=1, batch_norm=False,
                 dropout=0.0, activation=nn.ReLU):
        super().__init__()
        norm1 = nn.BatchNorm2d(out_channels) if batch_norm else nn.Identity()
        norm2 = nn.BatchNorm2d(out_channels) if batch_norm else nn.Identity()
        regularizer = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding),
            norm1,
            activation(),
            nn.Conv2d(out_channels, out_channels, kernel_size, padding=padding),
            norm2,
            activation(),
            regularizer,
        )

    def forward(self, images):
        features = self.layers(images)
        return features

In [ ]:
example_block = ConvBlock(3, 16, stride=2, batch_norm=True, dropout=.2)
block_output = example_block(images[:2])
print(example_block)
print("Вход:", images[:2].shape, "-> выход:", block_output.shape)

## 7. Готовая модель из нескольких блоков

На каждом шаге число каналов умножаем на `channel_multiplier`. После блоков приводим каждую карту к сетке 4×4 через AdaptiveAvgPool. В голове сохраняются признаки разных областей картинки; `pool_size=1` оставит только одно число на канал. `head_sizes` задает скрытые размеры полносвязной головы; пустой кортеж оставит один выходной Linear. `activation` передаем как класс, чтобы в каждом месте создать отдельный модуль.

In [ ]:
class ConvClassifier(nn.Module):
    def __init__(self, in_channels=3, num_classes=10, num_blocks=2,
                 base_channels=16, channel_multiplier=2, head_sizes=(64,), pool_size=4,
                 stride=2, batch_norm=False, dropout=0.0, activation=nn.ReLU):
        super().__init__()
        blocks = []
        current_channels = in_channels

        for index in range(num_blocks):
            out_channels = base_channels * channel_multiplier ** index
            block = ConvBlock(current_channels, out_channels, stride=stride,
                              batch_norm=batch_norm, dropout=dropout, activation=activation)
            blocks.append(block)
            current_channels = out_channels

        self.blocks = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool2d(pool_size)
        current_channels = current_channels * pool_size * pool_size
        head = [nn.Flatten()]
        for hidden_size in head_sizes:
            head.append(nn.Linear(current_channels, hidden_size))
            head.append(activation())
            current_channels = hidden_size
        head.append(nn.Linear(current_channels, num_classes))
        self.head = nn.Sequential(*head)

    def forward(self, images):
        features = self.blocks(images)
        features = self.pool(features)
        logits = self.head(features)
        return logits

In [ ]:
BASE = {"num_blocks": 3, "base_channels": 16, "head_sizes": (64,)}
base_example = ConvClassifier(**BASE)
features = images[:2]
shapes = []
for index, block in enumerate(base_example.blocks):
    features = block(features)
    shapes.append({"блок": index + 1, "форма выхода": tuple(features.shape)})
display(pd.DataFrame(shapes))
print("Выход модели:", base_example(images[:2]).shape)

from support.week11_utils import draw_pipeline
shape_labels = ["Вход\nNx3x32x32"]
for row in shapes:
    dimensions = "x".join(map(str, row["форма выхода"][1:]))
    shape_labels.append(f"Блок {row['блок']}\nNx{dimensions}")
shape_labels.extend(["Pool + Flatten\nNx1024", "Голова\nNx10"])
draw_pipeline(shape_labels, "Одна картинка -> карты признаков -> оценки десяти классов")

Для сравнения с предыдущим занятием используем готовую полносвязную сеть (MLP). Она разворачивает RGB-картинку в 3072 признака и использует ту же функцию потерь, разбиение и число эпох.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(), nn.Linear(3072, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 10),
        )

    def forward(self, images):
        logits = self.layers(images)
        return logits

## 8. Новый Dataset с аугментациями

При каждом чтении train-картинки случайно выбираем crop, отражение и изменение яркости. Это разные варианты одного объекта с той же меткой. Случайные преобразования выполняем **до нормализации**. В validation и test преобразования остаются детерминированными.

In [ ]:
class AugmentedCIFARDataset(CIFARDataset):
    def __init__(self, images, labels, transform):
        super().__init__(images, labels)
        self.transform = transform

    def __getitem__(self, index):
        image = torch.from_numpy(self.images[index].copy()).permute(2, 0, 1)
        image = self.transform(image)
        label = self.labels[index]
        return image, label

visual_transform = v2.Compose([
    v2.RandomCrop(32, padding=4, padding_mode="reflect"),
    v2.RandomHorizontalFlip(.5),
    v2.ColorJitter(brightness=.3, contrast=.3),
])
augmentation_transform = v2.Compose([
    visual_transform,
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean.tolist(), std.tolist()),
])
augmented_dataset = AugmentedCIFARDataset(train_images, train_labels, augmentation_transform)

In [ ]:
raw_rgb = train_images[0]
raw_chw = torch.from_numpy(raw_rgb.copy()).permute(2, 0, 1)
variants = [raw_rgb]
for _ in range(5):
    changed = visual_transform(raw_chw)
    variants.append(changed.permute(1, 2, 0).numpy())
show_images(variants, ["Исходная"] + ["torchvision"] * 5)

Те же преобразования можно собрать другими библиотеками. Albumentations принимает RGB-массив HWC через `image=...`. В OpenCV отдельные операции вызываем сами и сами выбираем их случайные параметры. Эти два примера нужны для сравнения интерфейсов; обучающий Dataset выше использует torchvision.

In [ ]:
import albumentations as A

albumentations_transform = A.Compose([
    A.HorizontalFlip(p=.5),
    A.Affine(translate_percent=(-.12, .12), scale=(.9, 1.1), rotate=(-8, 8), p=1),
    A.RandomBrightnessContrast(brightness_limit=.3, contrast_limit=.3, p=1),
], seed=SEED)
albumentations_images = [raw_rgb]
for _ in range(5):
    changed = albumentations_transform(image=raw_rgb)["image"]
    albumentations_images.append(changed)
show_images(albumentations_images, ["Исходная"] + ["Albumentations"] * 5)

In [ ]:
import cv2

rng = np.random.default_rng(SEED)
opencv_images = [raw_rgb]
for _ in range(5):
    shifted = np.float32([[1, 0, rng.integers(-4, 5)], [0, 1, rng.integers(-4, 5)]])
    changed = cv2.warpAffine(raw_rgb, shifted, (32, 32), borderMode=cv2.BORDER_REFLECT_101)
    if rng.random() < .5:
        changed = cv2.flip(changed, 1)
    brightness = rng.uniform(.7, 1.3)
    changed = np.clip(changed.astype(float) * brightness, 0, 255).astype(np.uint8)
    opencv_images.append(changed)
show_images(opencv_images, ["Исходная"] + ["OpenCV"] * 5)

## 9. Какие эксперименты запускаем

Базовая CNN имеет три блока с каналами 16 → 32 → 64 и голову 1024 → 64 → 10. BatchNorm, Dropout и аугментации проверяем отдельно, затем смотрим совместное действие BatchNorm и Dropout. У small/large меняется емкость модели: ширина, число блоков и голова. Все варианты получают одинаковый бюджет в восемь эпох; он не равен одинаковому времени.

Adam сглаживает градиент и масштабирует его по координатам; SGD с momentum накапливает направление; Нестеров добавляет поправку на инерционный сдвиг; AdamW отделяет уменьшение весов от моментов градиента. Learning rate для SGD равен 0.03, для Adam/AdamW — 0.001. Weight decay равен 0.0001. Здесь сравниваем **эти настройки**, без заявления об оптимальности каждого метода.

In [ ]:
optimizer_settings = {
    "Adam": (torch.optim.Adam, {"lr": .001, "weight_decay": 1e-4}),
    "SGD": (torch.optim.SGD, {"lr": .03, "momentum": .9, "weight_decay": 1e-4}),
    "Nesterov": (torch.optim.SGD, {"lr": .03, "momentum": .9, "nesterov": True, "weight_decay": 1e-4}),
    "AdamW": (torch.optim.AdamW, {"lr": .001, "weight_decay": 1e-4}),
}

# Имя, параметры CNN (None означает MLP), оптимизатор, аугментации.
experiments = [
    ("MLP", None, "Adam", False),
    ("CNN base", BASE, "Adam", False),
    ("CNN + BN", {**BASE, "batch_norm": True}, "Adam", False),
    ("CNN + Dropout", {**BASE, "dropout": .2}, "Adam", False),
    ("CNN + BN + Dropout", {**BASE, "batch_norm": True, "dropout": .2}, "Adam", False),
    ("CNN small", {"num_blocks": 2, "base_channels": 8, "head_sizes": (32,)}, "Adam", False),
    ("CNN large", {"num_blocks": 3, "base_channels": 32, "head_sizes": (128,)}, "Adam", False),
    ("CNN + aug", BASE, "Adam", True),
    ("CNN + SGD", BASE, "SGD", False),
    ("CNN + Nesterov", BASE, "Nesterov", False),
    ("CNN + AdamW", BASE, "AdamW", False),
]
configuration_rows = []
for name, parameters, optimizer_name, augment in experiments:
    row = {"модель": name, "optimizer": optimizer_name, "aug": augment}
    row.update(parameters or {"архитектура": "MLP"})
    configuration_rows.append(row)
display(pd.DataFrame(configuration_rows).fillna("-"))

## 10. Обучаем модели и смотрим кривые

Перед каждым запуском восстанавливаем seed и генератор порядка train. При одинаковой форме сети общие начальные веса совпадают. Все модели используют написанную выше `train_epoch`; лучшую эпоху выбираем по validation-loss.

Графики разделяем на четыре небольшие группы. Train-loss нужен для оценки подгонки, validation-loss — для выбора эпохи; разрыв между ними помогает заметить переобучение. При Dropout и аугментациях train проходит в более сложных условиях, поэтому его loss может оказаться выше validation-loss.

In [ ]:
for name, parameters, optimizer_name, augment in experiments:
    seed_everything(SEED)
    model = MLP() if parameters is None else ConvClassifier(**parameters)
    model = model.to(DEVICE)
    optimizer_class, optimizer_parameters = optimizer_settings[optimizer_name]
    optimizer = optimizer_class(model.parameters(), **optimizer_parameters)

    dataset = augmented_dataset if augment else train_dataset
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        generator=generator, num_workers=0)
    print()
    print(name)
    history, seconds = fit_model(model, loader, optimizer)
    models[name] = model
    histories[name] = history
    training_times[name] = seconds

In [ ]:
curve_groups = {
    "Архитектура и размер": ["Sequential", "MLP", "CNN small", "CNN base", "CNN large"],
    "BatchNorm и Dropout": ["CNN base", "CNN + BN", "CNN + Dropout", "CNN + BN + Dropout"],
    "Оптимизаторы": ["CNN base", "CNN + SGD", "CNN + Nesterov", "CNN + AdamW"],
    "Аугментации": ["CNN base", "CNN + aug"],
}
plot_training_groups(histories, curve_groups)

## 11. Одна таблица качества и скорости

Теперь вычисляем test-метрики всех выбранных моделей. **Accuracy** — доля верных ответов; **macro-F1** — среднее F1 десяти классов; **balanced accuracy** — средний recall классов. На сбалансированном test последняя совпадает с accuracy. **Top-3** учитывает попадание метки в три наибольшие оценки. Например, accuracy 0.60 означает 1200 верных ответов из 2000.

**Log loss** штрафует низкую вероятность истинного класса; **Brier** суммирует квадраты разностей вероятностей и one-hot метки. Для обоих меньше — лучше. Эти метрики различают модели с одинаковым argmax, но разной уверенностью.

Время обучения включает восемь эпох, validation и сохранение лучшей модели. Задержку batch=1 и число картинок/с при batch=128 измеряем после прогрева с синхронизацией устройства. В замер forward не входят загрузка картинок и преобразования. Параметры показывают объем модели, но сами по себе не определяют скорость.

In [ ]:
rows = []
probabilities_by_model = {}
for name, model in models.items():
    row, probabilities = evaluate_model(name, model, test_loader, DEVICE, training_times[name])
    rows.append(row)
    probabilities_by_model[name] = probabilities

comparison_table = pd.DataFrame(rows).set_index("model")
comparison_style = show_comparison(comparison_table, DEVICE)
plot_comparison(comparison_table)

### Качество при ограниченном времени

На карте ниже каждая точка — один фактический запуск. Наведение показывает метрики и стоимость; кнопки меняют горизонтальную ось между временем обучения, задержкой forward и числом параметров. Предпочтительны точки выше и левее. Размер точки приблизительно отражает число параметров.

Сравнение относится к указанным восьми эпохам и данному устройству. Изменение архитектуры может уменьшить число весов и одновременно увеличить задержку: это разные измерения.

In [ ]:
from pathlib import Path
from IPython.display import FileLink
from support.lesson11_visuals import export_tradeoff, plot_tradeoff_preview

plot_tradeoff_preview(comparison_table)
tradeoff_plot = export_tradeoff(comparison_table, Path("outputs/seminar_11_quality_cost.html"))
display(FileLink("outputs/seminar_11_quality_cost.html"))

В сохраненном прогоне небольшая Sequential дает около 59% accuracy против 48% у MLP, используя примерно в 12 раз меньше параметров. Готовая модель из блоков здесь не превосходит простую CNN: более глубокая архитектура сама по себе не гарантирует лучший результат. BatchNorm улучшает базовую модель; Dropout отдельно за восемь эпох не дает выигрыша.

Сопоставьте эти наблюдения с train/validation-кривыми. Затем смотрите парные изменения относительно CNN base. Первое место в одной метрике не обязательно означает первое место в другой. Разницу времени одного короткого запуска рассматриваем вместе с устройством и размером батча.

In [ ]:
class_metrics = show_class_metrics(test_labels, probabilities_by_model["CNN base"],
                                   "CNN base: какие классы путаются")

## 12. Проверяем аугментации на испорченных картинках

CNN base и CNN + aug отличаются только преобразованиями train. Обеим подаем одинаковый test: чистые картинки, сдвиг на 4 пикселя, затемнение до 60%, шум σ=0.08 и размытие σ=1. Нормализацию применяем после искажения. Шум и размытие в обучающие аугментации не входили.

Положительная разность accuracy означает выигрыш модели с аугментациями. Улучшение при сдвиге и освещении ожидается, но его размер проверяем по данным; устойчивость к шуму не следует автоматически из RandomCrop или ColorJitter.

In [ ]:
conditions = ["clean", "shift", "brightness", "noise", "blur"]
examples = [corrupt_images(test_images[:1], kind, severity=2)[0] for kind in conditions]
show_images(examples, conditions, columns=5)

robustness_rows = []
for kind in conditions:
    changed_images = corrupt_images(test_images, kind, severity=2, seed=123)
    changed_dataset = CIFARDataset(changed_images, test_labels)
    changed_loader = DataLoader(changed_dataset, batch_size=BATCH_SIZE, shuffle=False)

    for name in ["CNN base", "CNN + aug"]:
        probabilities = predict_probabilities(models[name], changed_loader, DEVICE)
        metrics = classification_metrics(test_labels, probabilities)
        robustness_rows.append({"condition": kind, "model": name, **metrics})

robustness_table = pd.DataFrame(robustness_rows)
robustness_accuracy = robustness_table.pivot(index="condition", columns="model", values="accuracy")
robustness_accuracy = robustness_accuracy.reindex(conditions)
robustness_accuracy["выигрыш, п.п."] = 100 * (robustness_accuracy["CNN + aug"] - robustness_accuracy["CNN base"])
display(robustness_accuracy.style.format({"CNN base": "{:.1%}", "CNN + aug": "{:.1%}",
                                        "выигрыш, п.п.": "{:+.2f}"}))
plot_robustness(robustness_accuracy)

В сохраненном прогоне аугментации улучшают accuracy при сдвиге и размытии, но не при затемнении или шуме. На чистом test модель с аугментациями тоже слабее: обучение на измененных картинках за восемь эпох оказалось более трудной задачей. Это видно и по train/validation-кривым.

По каждой строке сравните две модели и падение относительно чистых картинок. Если выигрыша нет, проверьте, встречалось ли подобное изменение при обучении. Один seed и восемь эпох показывают конкретный опыт; для устойчивого вывода нужны повторные обучения. Новые настройки выбираем по validation, сохраняя test для отчета.

## 13. Разные размеры: resize изображения или pooling?

В CIFAR-10 все исходные картинки 32×32. Для контролируемого опыта задаём каждому объекту один из размеров **32×32, 32×48, 48×32, 48×48** и детерминированно масштабируем его. Это имитация разных разрешений и пропорций; новых деталей в картинках не появляется. Разбиение и метки остаются прежними.

Сравним три способа обработки одного и того же подготовленного изображения:

| Вариант | До свёрток | Перед Linear |
|:--|:--|:--|
| Resize входа | bilinear resize до 32×32, antialias=True | AdaptiveAvgPool2d(4×4) |
| AvgPool входа | AdaptiveAvgPool2d(32×32) усредняет пиксели | AdaptiveAvgPool2d(4×4) |
| AvgPool признаков | исходный подготовленный размер | AdaptiveAvgPool2d(4×4) усредняет карты признаков |

Архитектура во всех случаях — одна ConvClassifier с BASE: три блока, голова 1024→64→10. Веса и их количество совпадают на старте. При фиксированном входе 32×32 после блоков уже получается 4×4: pooling головы не меняет эту форму. У третьего варианта размер карт зависит от входа, а pooling приводит их к той же голове.

Случайные аугментации выключены. Восемь эпох, Adam с теми же настройками, одинаковые seed и порядок объектов. В одном батче собираем только изображения с одинаковым подготовленным размером. **Эту же группировку и порядок батчей используем для всех трёх вариантов**, даже если предобработка уже сделала входы одинаковыми. Так не меняется состав мини-батчей.

В используемой версии MPS `AdaptiveAvgPool2d` не поддерживает неделимые размеры, например карту 6×6 с выходом 4×4. Поэтому при DEVICE=mps **эта секция выполняет все три варианта на CPU**; на CPU/CUDA сохраняет выбранное устройство. Устройство указано в отчёте о времени.

In [ ]:
from torch.utils.data import Sampler, Subset
from torch.nn import functional as F
from support.week11_utils import benchmark_inference, parameter_count, plot_metric_panels

RESOLUTION_DEVICE = torch.device("cpu") if DEVICE.type == "mps" else DEVICE
print("Устройство сравнения размеров:", RESOLUTION_DEVICE)

RESOLUTION_SHAPES = [(32, 32), (32, 48), (48, 32), (48, 48)]
RESOLUTION_POLICIES = {"Resize входа": "resize", "AvgPool входа": "input_pool", "AvgPool признаков": "feature_pool"}

# Один и тот же размер закреплён за объектом во всех вариантах.
resolution_groups = {}
for offset, (split, labels) in enumerate([("train", train_labels), ("val", val_labels), ("test", test_labels)]):
    generator = np.random.default_rng(SEED + offset)
    resolution_groups[split] = generator.integers(0, len(RESOLUTION_SHAPES), size=len(labels))

class VariableResolutionCIFAR(Dataset):
    def __init__(self, images, labels, groups, policy):
        self.images = images
        self.labels = torch.as_tensor(labels, dtype=torch.long)
        self.groups = np.asarray(groups)
        self.policy = policy

    def __len__(self):
        length = len(self.labels)
        return length

    def simulated_image(self, index):
        image = torch.from_numpy(self.images[index].copy()).permute(2, 0, 1).float() / 255
        size = RESOLUTION_SHAPES[self.groups[index]]
        image = F.interpolate(image[None], size=size, mode="bilinear", align_corners=False, antialias=True)[0]
        return image

    def prepare_image(self, image):
        if self.policy == "resize":
            image = F.interpolate(image[None], size=(32, 32), mode="bilinear", align_corners=False, antialias=True)[0]
        elif self.policy == "input_pool":
            image = F.adaptive_avg_pool2d(image, (32, 32))
        normalized = (image - MEAN) / STD
        return normalized

    def __getitem__(self, index):
        image = self.simulated_image(index)
        image = self.prepare_image(image)
        label = self.labels[index]
        return image, label

class ResolutionBatchSampler(Sampler):
    def __init__(self, groups, batch_size, shuffle, seed=SEED):
        self.groups = np.asarray(groups)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.generator = torch.Generator().manual_seed(seed)

    def __len__(self):
        counts = np.bincount(self.groups, minlength=len(RESOLUTION_SHAPES))
        length = int(np.sum((counts + self.batch_size - 1) // self.batch_size))
        return length

    def __iter__(self):
        batches = []
        for group in range(len(RESOLUTION_SHAPES)):
            indices = np.flatnonzero(self.groups == group)
            if self.shuffle:
                order = torch.randperm(len(indices), generator=self.generator).numpy()
                indices = indices[order]
            for start in range(0, len(indices), self.batch_size):
                batches.append(indices[start:start + self.batch_size].tolist())
        if self.shuffle:
            order = torch.randperm(len(batches), generator=self.generator).tolist()
            batches = [batches[index] for index in order]
        yield from batches

# Проверяем, что группировка не теряет объекты и одинакова у разных вариантов.
reference_batches = list(ResolutionBatchSampler(resolution_groups["train"], BATCH_SIZE, True))
assert sorted(index for batch in reference_batches for index in batch) == list(range(len(train_images)))
assert all(len(set(resolution_groups["train"][batch])) == 1 for batch in reference_batches)
assert reference_batches == list(ResolutionBatchSampler(resolution_groups["train"], BATCH_SIZE, True))

In [ ]:
from support.lesson11_visuals import plot_resolution_examples

example_index = int(np.flatnonzero(resolution_groups["train"] == 1)[0])
resolution_examples = {"Исходный CIFAR-10": train_images[example_index] / 255}
for name, policy in RESOLUTION_POLICIES.items():
    dataset = VariableResolutionCIFAR(train_images, train_labels, resolution_groups["train"], policy)
    normalized, _ = dataset[example_index]
    visible = (normalized * STD + MEAN).permute(1, 2, 0).numpy()
    resolution_examples[name] = np.clip(visible, 0, 1)
plot_resolution_examples(resolution_examples);

In [ ]:
def make_resolution_loader(images, labels, groups, policy, shuffle):
    dataset = VariableResolutionCIFAR(images, labels, groups, policy)
    sampler = ResolutionBatchSampler(groups, BATCH_SIZE, shuffle, seed=SEED)
    loader = DataLoader(dataset, batch_sampler=sampler, num_workers=0)
    return loader


def fit_resolution_model(model, train_loader, validation_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), **optimizer_settings["Adam"][1])
    best_loss = float("inf")
    best_state = None
    history = []
    synchronize(RESOLUTION_DEVICE)
    started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, RESOLUTION_DEVICE)
        val_loss, val_accuracy = validate(model, validation_loader, criterion, RESOLUTION_DEVICE)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_accuracy": val_accuracy})
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
        print(f"{epoch}: train={train_loss:.3f}, val={val_loss:.3f}, acc={val_accuracy:.3f}")
    model.load_state_dict(best_state)
    model.eval()
    synchronize(RESOLUTION_DEVICE)
    elapsed = time.perf_counter() - started
    result = (history, elapsed)
    return result


@torch.inference_mode()
def predict_resolution(model, loader):
    model.eval()
    probability_parts = []
    label_parts = []
    for images, labels in loader:
        logits = model(images.to(RESOLUTION_DEVICE))
        probability_parts.append(logits.softmax(dim=1).cpu().numpy())
        label_parts.append(labels.numpy())
    probabilities = np.concatenate(probability_parts)
    labels = np.concatenate(label_parts)
    result = (labels, probabilities)
    return result

In [ ]:
seed_everything(SEED)
resolution_initial_state = copy.deepcopy(ConvClassifier(**BASE).state_dict())
resolution_models = {}
resolution_histories = {}
resolution_train_seconds = {}
for name, policy in RESOLUTION_POLICIES.items():
    seed_everything(SEED)
    model = ConvClassifier(**BASE).to(RESOLUTION_DEVICE)
    model.load_state_dict(resolution_initial_state)
    assert all(torch.equal(value.cpu(), resolution_initial_state[key]) for key, value in model.state_dict().items())
    train_part = make_resolution_loader(train_images, train_labels, resolution_groups["train"], policy, True)
    validation_part = make_resolution_loader(val_images, val_labels, resolution_groups["val"], policy, False)
    print(name)
    history, seconds = fit_resolution_model(model, train_part, validation_part)
    resolution_models[name] = model
    resolution_histories[name] = history
    resolution_train_seconds[name] = seconds
plot_training_groups(resolution_histories, {"Разные размеры и предобработка": list(RESOLUTION_POLICIES)})

### Качество и стоимость трёх подходов

Эпоху выбираем по validation. Test оцениваем после обучения, сохраняя все три результата. Метки собираем вместе с предсказаниями: группировка по размеру изменила порядок объектов относительно исходного массива.

Время обучения включает подготовку картинок, train и validation. Задержки forward измеряем на уже подготовленном входе, отдельно для каждого исходного размера. В общей таблице показана **взвешенная по числу test-объектов сумма медиан** batch=1; картинки/с получены из такого же среднего времени batch=128. Это не медиана смеси всех запросов.

Предобработку отдельно измеряем на CPU: уже подготовленное изображение разного размера → выбранная операция → нормализация. Декодирование и создание имитированного разрешения в этот замер не входят. На чистом CIFAR-10 оба преобразования 32×32→32×32 почти тождественны; различия проверяем именно на одинаково подготовленных разных размерах.

In [ ]:
resolution_rows = []
resolution_size_rows = []
for name, policy in RESOLUTION_POLICIES.items():
    model = resolution_models[name]
    dataset = VariableResolutionCIFAR(test_images, test_labels, resolution_groups["test"], policy)
    label_parts = []
    probability_parts = []
    profiles = []
    for group, size in enumerate(RESOLUTION_SHAPES):
        indices = np.flatnonzero(resolution_groups["test"] == group)
        loader = DataLoader(Subset(dataset, indices.tolist()), batch_size=BATCH_SIZE, shuffle=False)
        labels, probabilities = predict_resolution(model, loader)
        label_parts.append(labels)
        probability_parts.append(probabilities)
        quality = classification_metrics(labels, probabilities)
        example, _ = next(iter(loader))
        assert len(example) == BATCH_SIZE
        single = benchmark_inference(model, example[:1], RESOLUTION_DEVICE, repeats=30, warmup=10)
        batch = benchmark_inference(model, example, RESOLUTION_DEVICE, repeats=30, warmup=10)
        raw_image = dataset.simulated_image(int(indices[0]))
        for _ in range(5):
            dataset.prepare_image(raw_image)
        preprocess_times = []
        for _ in range(30):
            started = time.perf_counter()
            dataset.prepare_image(raw_image)
            preprocess_times.append(1000 * (time.perf_counter() - started))
        profile = {"model": name, "размер до предобработки": f"{size[0]}x{size[1]}",
                   "размер входа CNN": f"{example.shape[2]}x{example.shape[3]}", "число объектов": len(indices),
                   "batch1_ms": single["latency_ms"], "batch128_ms": batch["latency_ms"],
                   "предобработка CPU, мс": float(np.median(preprocess_times)), **quality}
        resolution_size_rows.append(profile)
        profiles.append(profile)
    labels = np.concatenate(label_parts)
    probabilities = np.concatenate(probability_parts)
    assert len(labels) == len(test_labels)
    weights = np.array([row["число объектов"] for row in profiles])
    single_ms = np.average([row["batch1_ms"] for row in profiles], weights=weights)
    batch_ms = np.average([row["batch128_ms"] for row in profiles], weights=weights)
    preprocess_ms = np.average([row["предобработка CPU, мс"] for row in profiles], weights=weights)
    row = {"model": name, **classification_metrics(labels, probabilities), "parameters": parameter_count(model),
           "train_s": resolution_train_seconds[name], "batch1_ms": single_ms, "batch128_ms": batch_ms,
           "images_s": 1000 * BATCH_SIZE / batch_ms, "предобработка CPU, мс": preprocess_ms}
    resolution_rows.append(row)
resolution_table = pd.DataFrame(resolution_rows).set_index("model")
resolution_by_size = pd.DataFrame(resolution_size_rows).set_index(["model", "размер до предобработки"])
assert resolution_table.parameters.nunique() == 1
resolution_style = show_comparison(resolution_table, RESOLUTION_DEVICE)
display(resolution_table[["предобработка CPU, мс", "batch1_ms", "batch128_ms"]].style.format("{:.3f}"))
display(resolution_by_size[["размер входа CNN", "число объектов", "accuracy", "macro_f1", "log_loss", "batch1_ms",
                            "предобработка CPU, мс"]].style.format({"accuracy": "{:.1%}", "macro_f1": "{:.3f}",
                            "log_loss": "{:.3f}", "batch1_ms": "{:.3f}", "предобработка CPU, мс": "{:.3f}"}))
plot_metric_panels(resolution_table, title="Три способа работы с размером  |  один test")

In [ ]:
from IPython.display import Markdown

resize_row = resolution_table.loc["Resize входа"]
input_pool_row = resolution_table.loc["AvgPool входа"]
feature_pool_row = resolution_table.loc["AvgPool признаков"]
resolution_summary = (
    f"**Этот запуск:** seed={SEED}, {EPOCHS} эпох, {len(test_labels)} test-объектов, "
    f"устройство {RESOLUTION_DEVICE}. Accuracy: resize - {resize_row['accuracy']:.1%}, "
    f"AvgPool входа - {input_pool_row['accuracy']:.1%}, "
    f"AvgPool признаков - {feature_pool_row['accuracy']:.1%}. "
    f"Число параметров одинаково: {int(resize_row['parameters']):,}. "
    f"Обучение: {resize_row['train_s']:.1f}, {input_pool_row['train_s']:.1f}, "
    f"{feature_pool_row['train_s']:.1f} с соответственно. "
    f"Log loss: {resize_row['log_loss']:.3f}, {input_pool_row['log_loss']:.3f}, "
    f"{feature_pool_row['log_loss']:.3f}."
)
display(Markdown(resolution_summary))

### Как интерпретировать результат

Сравните accuracy, macro-F1 и log loss, затем стоимость. Если порядок методов меняется между метриками, это разные свойства ответов, а не ошибка таблицы. Посмотрите также отдельные размеры: одинаковое среднее может скрыть различия между квадратными и вытянутыми входами.

Входной AvgPool и bilinear resize по-разному усредняют пиксели; свёртки исходного размера видят другую сетку признаков и требуют других вычислений. Равное число весов не делает эти процедуры эквивалентными. По одному seed на искусственно изменённом CIFAR-10 нельзя установить универсальный лучший способ. Следующий опыт — несколько seed и данные с настоящими разными разрешениями; новые настройки выбирают по validation.